In [ ]:
"""
simulacion_cumulo.py

Simulador cinemático tipo Gaia para cúmulos abiertos.

El modelo genera miembros de un cúmulo abierto en 3D, asigna una
velocidad espacial común dirigida hacia un ápex y proyecta esa cinemática
a observables astrométricos tipo Gaia:

    ra        [deg]
    dec       [deg]
    parallax  [mas]
    pmra      [mas / yr]  = mu_alpha * cos(dec)
    pmdec     [mas / yr]

Incluye modelos reutilizables de error observacional:

    - error constante en paralaje
    - error constante en pmra / pmdec
    - error triangular en log10(pmra_error)
    - error triangular en log10(pmdec_error)
    - modelo Gaia-like acoplado:
        pmdec_error ≈ slope * pmra_error
        corr(dpmra, dpmdec) configurable

El simulador acepta el ápex en coordenadas ecuatoriales:

    apex_ra_deg, apex_dec_deg

o en coordenadas galácticas:

    apex_l_deg, apex_b_deg

pero no ambas al mismo tiempo.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
from astropy import units as u
from astropy.coordinates import SkyCoord


KM_S_PER_ARCSEC_YR_PC = 4.74047


# ============================================================
# Modelos de error reutilizables
# ============================================================

PMRA_LOG10_TRIANGULAR_ERROR = dict(
    pmra_error_distribution="log10_triangular",
    pmra_error_log10_left=-3.0,
    pmra_error_log10_mode=-1.7,
    pmra_error_log10_right=0.0,
)


PMDEC_LOG10_TRIANGULAR_ERROR = dict(
    pmdec_error_distribution="log10_triangular",
    pmdec_error_log10_left=-3.0,
    pmdec_error_log10_mode=-1.7,
    pmdec_error_log10_right=0.0,
)


GAIA_LIKE_PM_ERROR = dict(
    pmra_error_distribution="log10_triangular",
    pmra_error_log10_left=-3.0,
    pmra_error_log10_mode=-1.7,
    pmra_error_log10_right=0.0,

    # Relación empírica:
    # pmdec_error ≈ 0.7 * pmra_error
    pm_error_slope=0.7,

    # Correlación entre perturbaciones observacionales
    # dpmra y dpmdec.
    pmra_pmdec_corr=0.2,

    # Dispersión logarítmica alrededor de la pendiente.
    # 0.0 produce una línea exacta.
    pm_error_slope_logscatter=0.08,
)


@dataclass(frozen=True)
class ClusterSimulationConfig:
    """
    Configuración física y observacional de la simulación.

    Parameters
    ----------
    n_members : int
        Número de estrellas simuladas.

    center_ra_deg : float
        Ascensión recta del centro del cúmulo, en grados.

    center_dec_deg : float
        Declinación del centro del cúmulo, en grados.

    distance_pc : float
        Distancia heliocéntrica al centro del cúmulo, en parsec.

    radius_pc : float
        Radio físico máximo del cúmulo, en parsec.

    speed_kms : float
        Velocidad espacial total del cúmulo, en km/s.

    apex_ra_deg, apex_dec_deg : float, optional
        Dirección ecuatorial del ápex, en grados.

    apex_l_deg, apex_b_deg : float, optional
        Dirección galáctica del ápex, en grados.

    speed_sigma_kms : float, optional
        Dispersión física/intrínseca estrella-a-estrella en velocidad,
        en km/s. No es error observacional.

    apex_ra_sigma_deg, apex_dec_sigma_deg : float, optional
        Dispersión física/intrínseca del ápex en coordenadas ecuatoriales.

    apex_l_sigma_deg, apex_b_sigma_deg : float, optional
        Dispersión física/intrínseca del ápex en coordenadas galácticas.

    parallax_error_mas : float, optional
        Error observacional gaussiano constante aplicado a la paralaje.

    proper_motion_error_masyr : float, optional
        Compatibilidad hacia atrás. Si se usa, aplica el mismo error
        constante a pmra y pmdec, siempre que no se haya definido un
        modelo específico para cada componente.

    pmra_error_masyr, pmdec_error_masyr : float, optional
        Errores constantes independientes para pmra y pmdec.

    pmra_error_distribution, pmdec_error_distribution : str
        Distribución para la incertidumbre observacional de cada componente.
        Opciones:

            "none"
            "constant"
            "log10_triangular"

    pmra_error_log10_left, pmra_error_log10_mode, pmra_error_log10_right : float
        Parámetros de la distribución triangular para log10(pmra_error).

    pmdec_error_log10_left, pmdec_error_log10_mode, pmdec_error_log10_right : float
        Parámetros de la distribución triangular para log10(pmdec_error).

    pm_error_slope : float or None
        Si no es None, activa el modelo Gaia-like:

            pmdec_error = pm_error_slope * pmra_error * scatter

        En ese caso, pmdec_error_distribution se ignora.

    pmra_pmdec_corr : float
        Correlación entre las perturbaciones observacionales dpmra y dpmdec.

    pm_error_slope_logscatter : float
        Dispersión logarítmica alrededor de la relación pmdec_error/pmra_error.

    position_error_mas : float, optional
        Error observacional gaussiano aplicado a ra y dec, en mas.

    seed : int or None, optional
        Semilla para reproducibilidad.

    include_true_values : bool
        Si True, agrega columnas con los valores sin ruido observacional.
    """

    n_members: int
    center_ra_deg: float
    center_dec_deg: float
    distance_pc: float
    radius_pc: float
    speed_kms: float

    # Ápex en coordenadas ecuatoriales
    apex_ra_deg: Optional[float] = None
    apex_dec_deg: Optional[float] = None

    # Ápex en coordenadas galácticas
    apex_l_deg: Optional[float] = None
    apex_b_deg: Optional[float] = None

    # Dispersión física/intrínseca
    speed_sigma_kms: float = 0.0

    apex_ra_sigma_deg: float = 0.0
    apex_dec_sigma_deg: float = 0.0

    apex_l_sigma_deg: float = 0.0
    apex_b_sigma_deg: float = 0.0

    # Errores observacionales
    parallax_error_mas: float = 0.0

    # Compatibilidad con versión antigua
    proper_motion_error_masyr: float = 0.0

    # Errores constantes independientes
    pmra_error_masyr: float = 0.0
    pmdec_error_masyr: float = 0.0

    # Distribuciones independientes de incertidumbre
    pmra_error_distribution: str = "constant"
    pmdec_error_distribution: str = "constant"

    # Triangular para log10(pmra_error)
    pmra_error_log10_left: float = -3.0
    pmra_error_log10_mode: float = -1.7
    pmra_error_log10_right: float = 0.0

    # Triangular para log10(pmdec_error)
    pmdec_error_log10_left: float = -3.0
    pmdec_error_log10_mode: float = -1.7
    pmdec_error_log10_right: float = 0.0

    # Modelo Gaia-like acoplado
    pm_error_slope: Optional[float] = None
    pm_error_slope_logscatter: float = 0.0
    pmra_pmdec_corr: float = 0.0

    position_error_mas: float = 0.0
    seed: Optional[int] = None
    include_true_values: bool = True


class OpenClusterSimulator:
    """
    Simulador cinemático de cúmulos abiertos tipo Gaia.
    """

    def __init__(self, config: ClusterSimulationConfig) -> None:
        """
        Inicializa el simulador.
        """

        self.config = config
        self._validate_config()
        self.rng = np.random.default_rng(config.seed)

    def simulate(self) -> pd.DataFrame:
        """
        Ejecuta la simulación completa.

        Returns
        -------
        pd.DataFrame
            Catálogo simulado con columnas principales:

            - source_id
            - ra
            - dec
            - parallax
            - pmra
            - pmdec

            También incluye columnas verdaderas y diagnósticas.
        """

        positions_pc = self._build_cluster_positions()
        velocities_kms = self._build_cluster_velocities()

        catalog = self._phase_space_to_gaia_observables(
            positions_pc=positions_pc,
            velocities_kms=velocities_kms,
        )

        catalog.insert(
            loc=0,
            column="source_id",
            value=np.arange(1, self.config.n_members + 1),
        )

        if self.config.include_true_values:
            catalog["ra_true"] = catalog["ra"]
            catalog["dec_true"] = catalog["dec"]
            catalog["parallax_true"] = catalog["parallax"]
            catalog["pmra_true"] = catalog["pmra"]
            catalog["pmdec_true"] = catalog["pmdec"]

        catalog = self._apply_observational_noise(catalog)

        return catalog

    def _validate_config(self) -> None:
        """
        Valida que los parámetros sean físicamente razonables.
        """

        config = self.config

        if config.n_members <= 0:
            raise ValueError("n_members must be greater than zero.")

        if config.distance_pc <= 0.0:
            raise ValueError("distance_pc must be greater than zero.")

        if config.radius_pc < 0.0:
            raise ValueError("radius_pc must be non-negative.")

        if config.radius_pc >= config.distance_pc:
            raise ValueError(
                "radius_pc must be smaller than distance_pc. "
                "Otherwise some sources may have non-physical distances."
            )

        if config.speed_kms < 0.0:
            raise ValueError("speed_kms must be non-negative.")

        has_equatorial_apex = (
            config.apex_ra_deg is not None
            and config.apex_dec_deg is not None
        )

        has_galactic_apex = (
            config.apex_l_deg is not None
            and config.apex_b_deg is not None
        )

        incomplete_equatorial_apex = (
            config.apex_ra_deg is None
        ) != (
            config.apex_dec_deg is None
        )

        incomplete_galactic_apex = (
            config.apex_l_deg is None
        ) != (
            config.apex_b_deg is None
        )

        if incomplete_equatorial_apex:
            raise ValueError(
                "apex_ra_deg and apex_dec_deg must be provided together."
            )

        if incomplete_galactic_apex:
            raise ValueError(
                "apex_l_deg and apex_b_deg must be provided together."
            )

        if has_equatorial_apex and has_galactic_apex:
            raise ValueError(
                "Provide either equatorial apex "
                "(apex_ra_deg, apex_dec_deg) or galactic apex "
                "(apex_l_deg, apex_b_deg), but not both."
            )

        if not has_equatorial_apex and not has_galactic_apex:
            raise ValueError(
                "You must provide an apex direction using either "
                "(apex_ra_deg, apex_dec_deg) or "
                "(apex_l_deg, apex_b_deg)."
            )

        if has_equatorial_apex:
            if not (0.0 <= config.apex_ra_deg < 360.0):
                raise ValueError("apex_ra_deg must be in [0, 360).")

            if not (-90.0 <= config.apex_dec_deg <= 90.0):
                raise ValueError("apex_dec_deg must be in [-90, 90].")

        if has_galactic_apex:
            if not (0.0 <= config.apex_l_deg < 360.0):
                raise ValueError("apex_l_deg must be in [0, 360).")

            if not (-90.0 <= config.apex_b_deg <= 90.0):
                raise ValueError("apex_b_deg must be in [-90, 90].")

        sigma_values = {
            "speed_sigma_kms": config.speed_sigma_kms,
            "apex_ra_sigma_deg": config.apex_ra_sigma_deg,
            "apex_dec_sigma_deg": config.apex_dec_sigma_deg,
            "apex_l_sigma_deg": config.apex_l_sigma_deg,
            "apex_b_sigma_deg": config.apex_b_sigma_deg,
            "parallax_error_mas": config.parallax_error_mas,
            "proper_motion_error_masyr": config.proper_motion_error_masyr,
            "pmra_error_masyr": config.pmra_error_masyr,
            "pmdec_error_masyr": config.pmdec_error_masyr,
            "pm_error_slope_logscatter": config.pm_error_slope_logscatter,
            "position_error_mas": config.position_error_mas,
        }

        for name, value in sigma_values.items():
            if value < 0.0:
                raise ValueError(f"{name} must be non-negative.")

        valid_distributions = {"none", "constant", "log10_triangular"}

        if config.pmra_error_distribution not in valid_distributions:
            raise ValueError(
                "pmra_error_distribution must be one of "
                f"{valid_distributions}."
            )

        if config.pmdec_error_distribution not in valid_distributions:
            raise ValueError(
                "pmdec_error_distribution must be one of "
                f"{valid_distributions}."
            )

        if not (
            config.pmra_error_log10_left
            <= config.pmra_error_log10_mode
            <= config.pmra_error_log10_right
        ):
            raise ValueError(
                "pmra triangular parameters must satisfy "
                "left <= mode <= right."
            )

        if not (
            config.pmdec_error_log10_left
            <= config.pmdec_error_log10_mode
            <= config.pmdec_error_log10_right
        ):
            raise ValueError(
                "pmdec triangular parameters must satisfy "
                "left <= mode <= right."
            )

        if config.pm_error_slope is not None:
            if config.pm_error_slope < 0.0:
                raise ValueError(
                    "pm_error_slope must be non-negative or None."
                )

        if not (-1.0 <= config.pmra_pmdec_corr <= 1.0):
            raise ValueError("pmra_pmdec_corr must be between -1 and 1.")

    def _build_cluster_positions(self) -> np.ndarray:
        """
        Genera posiciones cartesianas heliocéntricas ICRS en pc.

        Returns
        -------
        np.ndarray
            Arreglo de forma (n_members, 3) con posiciones en pc.
        """

        config = self.config

        center_coord = SkyCoord(
            ra=config.center_ra_deg * u.deg,
            dec=config.center_dec_deg * u.deg,
            distance=config.distance_pc * u.pc,
            frame="icrs",
        )

        center_xyz_pc = center_coord.cartesian.xyz.to_value(u.pc)

        offsets_pc = self._sample_uniform_sphere(
            n_points=config.n_members,
            radius_pc=config.radius_pc,
        )

        return center_xyz_pc[None, :] + offsets_pc

    def _build_cluster_velocities(self) -> np.ndarray:
        """
        Genera velocidades cartesianas heliocéntricas ICRS en km/s.

        Returns
        -------
        np.ndarray
            Arreglo de forma (n_members, 3) con velocidades en km/s.
        """

        config = self.config

        speeds_kms = self.rng.normal(
            loc=config.speed_kms,
            scale=config.speed_sigma_kms,
            size=config.n_members,
        )
        speeds_kms = np.clip(speeds_kms, a_min=0.0, a_max=None)

        if config.apex_ra_deg is not None and config.apex_dec_deg is not None:
            apex_ra_deg = self.rng.normal(
                loc=config.apex_ra_deg,
                scale=config.apex_ra_sigma_deg,
                size=config.n_members,
            )
            apex_ra_deg = np.mod(apex_ra_deg, 360.0)

            apex_dec_deg = self.rng.normal(
                loc=config.apex_dec_deg,
                scale=config.apex_dec_sigma_deg,
                size=config.n_members,
            )
            apex_dec_deg = np.clip(apex_dec_deg, -90.0, 90.0)

            apex_unit_vectors = self._icrs_apex_to_icrs_unit_vector(
                apex_ra_deg=apex_ra_deg,
                apex_dec_deg=apex_dec_deg,
            )

        else:
            apex_l_deg = self.rng.normal(
                loc=config.apex_l_deg,
                scale=config.apex_l_sigma_deg,
                size=config.n_members,
            )
            apex_l_deg = np.mod(apex_l_deg, 360.0)

            apex_b_deg = self.rng.normal(
                loc=config.apex_b_deg,
                scale=config.apex_b_sigma_deg,
                size=config.n_members,
            )
            apex_b_deg = np.clip(apex_b_deg, -90.0, 90.0)

            apex_unit_vectors = self._galactic_apex_to_icrs_unit_vector(
                apex_l_deg=apex_l_deg,
                apex_b_deg=apex_b_deg,
            )

        return speeds_kms[:, None] * apex_unit_vectors

    def _sample_uniform_sphere(
        self,
        n_points: int,
        radius_pc: float,
    ) -> np.ndarray:
        """
        Muestrea puntos uniformemente dentro de una esfera 3D.

        Parameters
        ----------
        n_points : int
            Número de puntos.

        radius_pc : float
            Radio máximo de la esfera, en pc.

        Returns
        -------
        np.ndarray
            Offsets cartesianos de forma (n_points, 3) en pc.
        """

        if radius_pc == 0.0:
            return np.zeros((n_points, 3), dtype=float)

        directions = self.rng.normal(size=(n_points, 3))
        norms = np.linalg.norm(directions, axis=1)

        if np.any(norms == 0.0):
            raise RuntimeError("Random direction with zero norm generated.")

        unit_directions = directions / norms[:, None]
        radii = radius_pc * self.rng.random(n_points) ** (1.0 / 3.0)

        return unit_directions * radii[:, None]

    @staticmethod
    def _icrs_apex_to_icrs_unit_vector(
        apex_ra_deg: np.ndarray,
        apex_dec_deg: np.ndarray,
    ) -> np.ndarray:
        """
        Convierte direcciones de ápex ecuatoriales a vectores ICRS.

        Parameters
        ----------
        apex_ra_deg : np.ndarray
            Ascensiones rectas, en grados.

        apex_dec_deg : np.ndarray
            Declinaciones, en grados.

        Returns
        -------
        np.ndarray
            Vectores unitarios ICRS de forma (n_sources, 3).
        """

        apex_coords = SkyCoord(
            ra=apex_ra_deg * u.deg,
            dec=apex_dec_deg * u.deg,
            frame="icrs",
        )

        cartesian = apex_coords.cartesian

        return np.column_stack(
            (
                cartesian.x.value,
                cartesian.y.value,
                cartesian.z.value,
            )
        )

    @staticmethod
    def _galactic_apex_to_icrs_unit_vector(
        apex_l_deg: np.ndarray,
        apex_b_deg: np.ndarray,
    ) -> np.ndarray:
        """
        Convierte direcciones de ápex galácticas a vectores ICRS.

        Parameters
        ----------
        apex_l_deg : np.ndarray
            Longitudes galácticas, en grados.

        apex_b_deg : np.ndarray
            Latitudes galácticas, en grados.

        Returns
        -------
        np.ndarray
            Vectores unitarios ICRS de forma (n_sources, 3).
        """

        apex_coords = SkyCoord(
            l=apex_l_deg * u.deg,
            b=apex_b_deg * u.deg,
            frame="galactic",
        )

        icrs_cartesian = apex_coords.icrs.cartesian

        return np.column_stack(
            (
                icrs_cartesian.x.value,
                icrs_cartesian.y.value,
                icrs_cartesian.z.value,
            )
        )

    @staticmethod
    def _phase_space_to_gaia_observables(
        positions_pc: np.ndarray,
        velocities_kms: np.ndarray,
    ) -> pd.DataFrame:
        """
        Convierte fase espacial cartesiana ICRS a observables tipo Gaia.

        Parameters
        ----------
        positions_pc : np.ndarray
            Posiciones heliocéntricas cartesianas ICRS, en pc.
            Forma esperada: (n_sources, 3).

        velocities_kms : np.ndarray
            Velocidades heliocéntricas cartesianas ICRS, en km/s.
            Forma esperada: (n_sources, 3).

        Returns
        -------
        pd.DataFrame
            DataFrame con ra, dec, parallax, pmra y pmdec.
        """

        if positions_pc.shape != velocities_kms.shape:
            raise ValueError(
                "positions_pc and velocities_kms must have the same shape."
            )

        if positions_pc.ndim != 2 or positions_pc.shape[1] != 3:
            raise ValueError(
                "positions_pc and velocities_kms must have shape "
                "(n_sources, 3)."
            )

        x_coord = positions_pc[:, 0]
        y_coord = positions_pc[:, 1]
        z_coord = positions_pc[:, 2]

        distance_pc = np.linalg.norm(positions_pc, axis=1)

        if np.any(distance_pc <= 0.0):
            raise ValueError("All sources must have positive distance.")

        unit_position = positions_pc / distance_pc[:, None]

        ra_rad = np.arctan2(y_coord, x_coord) % (2.0 * np.pi)
        dec_rad = np.arcsin(np.clip(z_coord / distance_pc, -1.0, 1.0))

        sin_ra = np.sin(ra_rad)
        cos_ra = np.cos(ra_rad)
        sin_dec = np.sin(dec_rad)
        cos_dec = np.cos(dec_rad)

        basis_ra = np.column_stack(
            (
                -sin_ra,
                cos_ra,
                np.zeros_like(ra_rad),
            )
        )

        basis_dec = np.column_stack(
            (
                -cos_ra * sin_dec,
                -sin_ra * sin_dec,
                cos_dec,
            )
        )

        velocity_ra_kms = np.sum(velocities_kms * basis_ra, axis=1)
        velocity_dec_kms = np.sum(velocities_kms * basis_dec, axis=1)

        radial_velocity_kms = np.sum(
            velocities_kms * unit_position,
            axis=1,
        )

        pmra_masyr = (
            1_000.0
            * velocity_ra_kms
            / (KM_S_PER_ARCSEC_YR_PC * distance_pc)
        )

        pmdec_masyr = (
            1_000.0
            * velocity_dec_kms
            / (KM_S_PER_ARCSEC_YR_PC * distance_pc)
        )

        return pd.DataFrame(
            {
                "ra": np.degrees(ra_rad),
                "dec": np.degrees(dec_rad),
                "parallax": 1_000.0 / distance_pc,
                "pmra": pmra_masyr,
                "pmdec": pmdec_masyr,
                "distance_pc_true": distance_pc,
                "radial_velocity_kms_true": radial_velocity_kms,
            }
        )

    def _sample_error_sigma(
        self,
        distribution: str,
        constant_sigma: float,
        log10_left: float,
        log10_mode: float,
        log10_right: float,
        size: int,
        name: str,
    ) -> np.ndarray:
        """
        Genera incertidumbres observacionales sigma.

        Si distribution == "log10_triangular":

            log10(sigma) ~ Triangular(left, mode, right)

        La perturbación observacional final se aplica como:

            observable += Normal(0, sigma)
        """

        if distribution == "none":
            return np.zeros(size, dtype=float)

        if distribution == "constant":
            if constant_sigma <= 0.0:
                return np.zeros(size, dtype=float)

            return np.full(size, constant_sigma, dtype=float)

        if distribution == "log10_triangular":
            log10_sigma = self.rng.triangular(
                left=log10_left,
                mode=log10_mode,
                right=log10_right,
                size=size,
            )

            return 10.0**log10_sigma

        raise ValueError(f"Unknown error distribution for {name}: {distribution}")

    def _sample_proper_motion_errors(
        self,
        size: int,
    ) -> tuple[np.ndarray, np.ndarray]:
        """
        Genera incertidumbres observacionales para pmra y pmdec.

        Casos soportados:

        1. Solo pmra triangular:
            pmra_error_distribution="log10_triangular"

        2. Solo pmdec triangular:
            pmdec_error_distribution="log10_triangular"

        3. Gaia-like:
            pmra se genera con su distribución y
            pmdec_error ≈ pm_error_slope * pmra_error

        4. Compatibilidad antigua:
            proper_motion_error_masyr aplica error constante a ambas.
        """

        config = self.config

        pmra_constant = config.pmra_error_masyr
        pmdec_constant = config.pmdec_error_masyr

        # Compatibilidad con el modelo antiguo:
        # proper_motion_error_masyr aplica a ambas componentes,
        # salvo que el usuario haya definido algo específico.
        if config.proper_motion_error_masyr > 0.0:
            if (
                config.pmra_error_distribution == "constant"
                and pmra_constant == 0.0
            ):
                pmra_constant = config.proper_motion_error_masyr

            if (
                config.pmdec_error_distribution == "constant"
                and pmdec_constant == 0.0
                and config.pm_error_slope is None
            ):
                pmdec_constant = config.proper_motion_error_masyr

        pmra_error = self._sample_error_sigma(
            distribution=config.pmra_error_distribution,
            constant_sigma=pmra_constant,
            log10_left=config.pmra_error_log10_left,
            log10_mode=config.pmra_error_log10_mode,
            log10_right=config.pmra_error_log10_right,
            size=size,
            name="pmra",
        )

        # Modelo Gaia-like:
        # pmdec_error se deriva de pmra_error.
        if config.pm_error_slope is not None:
            if config.pm_error_slope_logscatter > 0.0:
                scatter_factor = 10.0 ** self.rng.normal(
                    loc=0.0,
                    scale=config.pm_error_slope_logscatter,
                    size=size,
                )
            else:
                scatter_factor = np.ones(size, dtype=float)

            pmdec_error = (
                config.pm_error_slope
                * pmra_error
                * scatter_factor
            )

        else:
            pmdec_error = self._sample_error_sigma(
                distribution=config.pmdec_error_distribution,
                constant_sigma=pmdec_constant,
                log10_left=config.pmdec_error_log10_left,
                log10_mode=config.pmdec_error_log10_mode,
                log10_right=config.pmdec_error_log10_right,
                size=size,
                name="pmdec",
            )

        return pmra_error, pmdec_error

    def _apply_observational_noise(
        self,
        catalog: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Aplica ruido observacional gaussiano a las columnas tipo Gaia.

        Parameters
        ----------
        catalog : pd.DataFrame
            Catálogo sin ruido observacional.

        Returns
        -------
        pd.DataFrame
            Catálogo con ruido observacional.
        """

        config = self.config
        result = catalog.copy()
        n_sources = len(result)

        # ------------------------------------------------------------
        # Error en posición
        # ------------------------------------------------------------
        if config.position_error_mas > 0.0:
            sigma_deg = config.position_error_mas / 3_600_000.0

            result["ra"] += self.rng.normal(
                loc=0.0,
                scale=sigma_deg,
                size=n_sources,
            )

            result["dec"] += self.rng.normal(
                loc=0.0,
                scale=sigma_deg,
                size=n_sources,
            )

            result["ra"] = np.mod(result["ra"], 360.0)
            result["dec"] = np.clip(result["dec"], -90.0, 90.0)

        # ------------------------------------------------------------
        # Error en paralaje
        # ------------------------------------------------------------
        if config.parallax_error_mas > 0.0:
            parallax_noise = self.rng.normal(
                loc=0.0,
                scale=config.parallax_error_mas,
                size=n_sources,
            )

            result["parallax"] += parallax_noise
            result["parallax_error"] = config.parallax_error_mas

        else:
            result["parallax_error"] = 0.0

        # ------------------------------------------------------------
        # Errores en movimientos propios
        # ------------------------------------------------------------
        pmra_error, pmdec_error = self._sample_proper_motion_errors(
            size=n_sources,
        )

        if np.any(pmra_error > 0.0) or np.any(pmdec_error > 0.0):
            rho = config.pmra_pmdec_corr

            z_pmra = self.rng.normal(
                loc=0.0,
                scale=1.0,
                size=n_sources,
            )

            z_independent = self.rng.normal(
                loc=0.0,
                scale=1.0,
                size=n_sources,
            )

            z_pmdec = (
                rho * z_pmra
                + np.sqrt(max(0.0, 1.0 - rho**2)) * z_independent
            )

            result["pmra"] += pmra_error * z_pmra
            result["pmdec"] += pmdec_error * z_pmdec

        result["pmra_error"] = pmra_error
        result["pmdec_error"] = pmdec_error

        return result


# ============================================================
# Ejemplos de configuración
# ============================================================

def build_example_configs() -> dict[str, ClusterSimulationConfig]:
    """
    Devuelve configuraciones de ejemplo para pruebas rápidas.
    """

    common_kwargs = dict(
        n_members=455,
        center_ra_deg=67.0,
        center_dec_deg=17.0,
        distance_pc=85.0,
        radius_pc=5.0,
        apex_ra_deg=97.89,
        apex_dec_deg=6.62,
        speed_kms=25.0,
        seed=42,
    )

    configs = {
        # Cúmulo ideal
        "ideal_hyades_apex": ClusterSimulationConfig(
            **common_kwargs,
        ),

        # Cúmulo ideal con otro ápex
        "ideal_solar_apex": ClusterSimulationConfig(
            n_members=455,
            center_ra_deg=67.0,
            center_dec_deg=17.0,
            distance_pc=85.0,
            radius_pc=5.0,
            apex_ra_deg=227.79628626384542,
            apex_dec_deg=-23.69031810716205,
            speed_kms=25.0,
            seed=42,
        ),

        # Solo error en paralaje
        "error_parallax": ClusterSimulationConfig(
            **common_kwargs,
            parallax_error_mas=0.03,
        ),

        # Solo error triangular en pmra
        "error_pmra_triangular": ClusterSimulationConfig(
            **common_kwargs,
            **PMRA_LOG10_TRIANGULAR_ERROR,
        ),

        # Solo error triangular en pmdec
        "error_pmdec_triangular": ClusterSimulationConfig(
            **common_kwargs,
            **PMDEC_LOG10_TRIANGULAR_ERROR,
        ),

        # Errores Gaia-like en pmra y pmdec
        "error_pm_gaia_like": ClusterSimulationConfig(
            **common_kwargs,
            **GAIA_LIKE_PM_ERROR,
        ),

        # Paralaje + Gaia-like
        "error_all_observational": ClusterSimulationConfig(
            **common_kwargs,
            parallax_error_mas=0.03,
            **GAIA_LIKE_PM_ERROR,
        ),

        # Dispersión física/intrínseca en velocidad
        "physical_velocity_dispersion": ClusterSimulationConfig(
            **common_kwargs,
            speed_sigma_kms=5.0,
        ),

        # Paralaje + Gaia-like + dispersión física
        "error_all_plus_dispersion": ClusterSimulationConfig(
            **common_kwargs,
            parallax_error_mas=0.03,
            speed_sigma_kms=5.0,
            **GAIA_LIKE_PM_ERROR,
        ),
    }

    return configs


if __name__ == "__main__":
    configs = build_example_configs()

    config = configs["error_pmdec_triangular"]

    simulator = OpenClusterSimulator(config)
    catalog = simulator.simulate()

    print(catalog.head())
    print()
    print(catalog[["pmra_error", "pmdec_error"]].describe())

    catalog.to_csv(
        "mock_open_cluster_gaia.csv",
        index=False,
    )

   source_id         ra       dec   parallax      pmra      pmdec  \
0          1  89.470478  1.303852  12.132140 -0.123624  63.962720   
1          2  88.702736 -1.795276  12.308138 -0.124059  64.881044   
2          3  89.411205 -0.077380  12.063140 -0.122818  63.617727   
3          4  91.826597  1.664711  11.367346 -0.119548  59.920373   
4          5  89.845199  1.095882  11.219983 -0.114926  59.158515   

   distance_pc_true  radial_velocity_kms_true    ra_true  dec_true  \
0         82.425690                  0.607555  89.470478  1.303852   
1         81.247057                 -0.743879  88.702736 -1.795276   
2         82.897157                  0.004990  89.411205 -0.077380   
3         87.971280                  0.762932  91.826597  1.664711   
4         89.126697                  0.516518  89.845199  1.095882   

   parallax_true  pmra_true  pmdec_true  
0      12.132140  -0.123624   63.962720  
1      12.308138  -0.124059   64.881044  
2      12.063140  -0.122818   63.61772

In [3]:
# ejemplo de uso
# from open_cluster_simulator import (
#     ClusterSimulationConfig,
#     OpenClusterSimulator,
# )

# config = ClusterSimulationConfig(
#     n_members=500,
#     center_ra_deg=186.25,
#     center_dec_deg=26.10,
#     distance_pc=85.0,
#     radius_pc=5.0,
#     apex_l_deg=180.0,
#     apex_b_deg=0.0,
#     speed_kms=25.0,
#     speed_sigma_kms=0.3,
#     apex_l_sigma_deg=1.0,
#     apex_b_sigma_deg=0.5,
#     parallax_error_mas=0.02,
#     proper_motion_error_masyr=0.05,
#     position_error_mas=0.1,
#     seed=42,
# )

# simulator = OpenClusterSimulator(config)
# df_cluster = simulator.simulate()

# df_cluster.head()